# Text Analytics — Document Preprocessing and TF-IDF

**Objective:**
1. Apply document preprocessing methods: **Tokenization, POS Tagging, Stop Words Removal, Stemming, Lemmatization**.
2. Create a vector representation of documents using **Term Frequency (TF)** and **Inverse Document Frequency (IDF)**.

### Why preprocess text?
Raw text contains noise — punctuation, common words ("the", "is"), inflections ("running" vs "ran"). Preprocessing converts text into a clean form so algorithms can compare documents fairly. Otherwise "running fast" and "ran fast" look like completely different sentences.

## Step 1: Install and Import Libraries

In [ ]:
# nltk — Natural Language Toolkit, the most popular text-processing library in Python.
import nltk

# Download the data NLTK needs (only required the first time you run this).
# punkt          — sentence/word tokenizer model
# stopwords      — list of common English stop words
# averaged_perceptron_tagger — POS (Part of Speech) tagger model
# wordnet        — lexical database for lemmatization
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

# Specific tools from NLTK
from nltk.tokenize import word_tokenize, sent_tokenize  # break text into words/sentences
from nltk.corpus   import stopwords                      # list of common stop words
from nltk.stem     import PorterStemmer, WordNetLemmatizer  # stemmer and lemmatizer
from nltk          import pos_tag                         # part-of-speech tagger

# pandas — for displaying TF-IDF results in a readable table
import pandas as pd

# sklearn — TF-IDF and Count vectorizers
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

print("All libraries imported successfully.")

## Step 2: Define a Sample Document

In [ ]:
# A sample paragraph that we will preprocess
document = """Text analytics is the process of deriving meaningful information from natural language text.
It involves several techniques such as tokenization, part of speech tagging, stemming, and lemmatization.
Text analytics is widely used in sentiment analysis, document classification, and information retrieval.
Modern systems use machine learning to analyze text efficiently and accurately."""

print("Original Document:")
print(document)

## Step 3: Tokenization

**Tokenization** = breaking text into smaller units called **tokens**. We do it at two levels:
- **Sentence tokenization** — splits text into sentences.
- **Word tokenization** — splits each sentence into individual words and punctuation.

In [ ]:
# Sentence tokenization — splits the document into individual sentences
sentences = sent_tokenize(document)

print(f"Number of sentences: {len(sentences)}")
for i, s in enumerate(sentences, 1):
    print(f"{i}. {s}")

In [ ]:
# Word tokenization — splits the document into words and punctuation marks
words = word_tokenize(document)

print(f"Number of tokens (words + punctuation): {len(words)}")
print("First 25 tokens:", words[:25])

## Step 4: POS (Part-of-Speech) Tagging

**POS Tagging** assigns each word a grammatical category — noun, verb, adjective, etc.

Common Penn Treebank tags:
- **NN** — Noun (e.g., "text")
- **VB / VBZ / VBG** — Verb / verb 3rd-person / gerund
- **JJ** — Adjective
- **RB** — Adverb
- **DT** — Determiner ("the", "a")
- **IN** — Preposition / conjunction

In [ ]:
# pos_tag() takes a list of tokens and returns a list of (word, POS_tag) tuples
pos_tags = pos_tag(words)

print("First 20 word-POS pairs:")
for word, tag in pos_tags[:20]:
    print(f"  {word:<20} -> {tag}")

## Step 5: Stop Words Removal

**Stop words** are extremely common words ("the", "is", "and", "of") that appear in almost every document. They carry little distinguishing information, so we usually remove them before analysis.

NLTK ships with a built-in list of English stop words.

In [ ]:
# Load the English stop word list
stop_words = set(stopwords.words('english'))

print(f"Total English stop words available: {len(stop_words)}")
print("Sample stop words:", list(stop_words)[:20])

In [ ]:
# Filter out stop words AND punctuation from our tokens.
# We lowercase first so 'The' is recognized as a stop word.
# isalpha() keeps only alphabetic tokens (drops punctuation like ',' and '.').
filtered_words = [w for w in words if w.lower() not in stop_words and w.isalpha()]

print(f"Tokens before filtering: {len(words)}")
print(f"Tokens after filtering : {len(filtered_words)}")
print("\nFiltered tokens:")
print(filtered_words)

## Step 6: Stemming

**Stemming** chops off word endings to reduce a word to its **stem** (root form).
- "running" → "run"
- "studies" → "studi"
- "happily" → "happili"

Stemming is fast but crude — the resulting stem isn't always a real word. Most popular algorithm: **Porter Stemmer**.

In [ ]:
# Create the Porter Stemmer
stemmer = PorterStemmer()

# Apply stemmer to each filtered word
stemmed_words = [stemmer.stem(w) for w in filtered_words]

# Show a side-by-side comparison
print(f"{'Original':<20} {'Stemmed':<20}")
print("-" * 40)
for original, stemmed in list(zip(filtered_words, stemmed_words))[:15]:
    print(f"{original:<20} {stemmed:<20}")

## Step 7: Lemmatization

**Lemmatization** also reduces words to their base form, but unlike stemming it uses a vocabulary and morphological rules to return an actual dictionary word (called the **lemma**).
- "running" → "run"
- "studies" → "study"
- "better" → "good"

Lemmatization is slower but more accurate. We use NLTK's `WordNetLemmatizer`.

In [ ]:
# Create the lemmatizer
lemmatizer = WordNetLemmatizer()

# Lemmatize each filtered word.
# By default it assumes 'noun'. Passing pos='v' lemmatizes as verb (running -> run).
lemmatized_words = [lemmatizer.lemmatize(w, pos='v') for w in filtered_words]

print(f"{'Original':<20} {'Lemmatized':<20}")
print("-" * 40)
for original, lemma in list(zip(filtered_words, lemmatized_words))[:15]:
    print(f"{original:<20} {lemma:<20}")

In [ ]:
# Compare stemming vs lemmatization side by side
comparison = pd.DataFrame({
    'Original'   : filtered_words,
    'Stemmed'    : stemmed_words,
    'Lemmatized' : lemmatized_words
})
print("Stemming vs Lemmatization:")
comparison.head(15)

## Step 8: Term Frequency (TF) and Inverse Document Frequency (IDF)

To do anything quantitative with text (similarity search, classification, clustering), we have to convert words into numbers. **TF-IDF** is the most common scheme.

### Term Frequency (TF)
> *How often does a term appear in a document?*

$$TF(t, d) = \frac{\text{count of } t \text{ in document } d}{\text{total terms in document } d}$$

### Inverse Document Frequency (IDF)
> *How rare is the term across all documents?*

$$IDF(t) = \log\left(\frac{N}{df(t)}\right)$$
where N = total number of documents, df(t) = number of documents containing t.

A term that appears in every document gets a low IDF. A term that appears in only one document gets a high IDF.

### TF-IDF
$$TF\text{-}IDF(t, d) = TF(t, d) \times IDF(t)$$

A high TF-IDF value means the term is **frequent in this document** but **rare across the corpus** — i.e., it characterizes this document. That's exactly what we want for distinguishing documents.

### 8.1 Build a Small Corpus

In [ ]:
# To compute IDF we need MULTIPLE documents.
# We treat each sentence of our paragraph as a separate document.
documents = [
    "Text analytics is the process of deriving meaningful information from natural language text.",
    "It involves several techniques such as tokenization, part of speech tagging, stemming, and lemmatization.",
    "Text analytics is widely used in sentiment analysis, document classification, and information retrieval.",
    "Modern systems use machine learning to analyze text efficiently and accurately."
]

print(f"Number of documents in the corpus: {len(documents)}")
for i, doc in enumerate(documents, 1):
    print(f"\nDocument {i}: {doc}")

### 8.2 Term Frequency using CountVectorizer

In [ ]:
# CountVectorizer counts occurrences of each word across all documents.
# stop_words='english' automatically removes common stop words.
count_vectorizer = CountVectorizer(stop_words='english')

# fit_transform() learns the vocabulary AND produces a document-term count matrix.
# Rows = documents, columns = terms, cell = count of that term in that document.
count_matrix = count_vectorizer.fit_transform(documents)

# Display as a DataFrame for readability
tf_df = pd.DataFrame(
    count_matrix.toarray(),
    columns=count_vectorizer.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(documents))]
)

print("Term Frequency Matrix (raw counts):")
tf_df

### 8.3 TF-IDF using TfidfVectorizer

In [ ]:
# TfidfVectorizer combines TF and IDF in one step.
tfidf_vectorizer = TfidfVectorizer(stop_words='english')

# fit_transform() returns the TF-IDF matrix
tfidf_matrix = tfidf_vectorizer.fit_transform(documents)

# Display as a DataFrame
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray().round(4),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(documents))]
)

print("TF-IDF Matrix:")
tfidf_df

In [ ]:
# Show just the IDF score per term — measures how rare each term is across documents.
# Higher IDF = appears in fewer documents = more distinctive.
idf_scores = pd.DataFrame({
    'Term': tfidf_vectorizer.get_feature_names_out(),
    'IDF':  tfidf_vectorizer.idf_.round(4)
}).sort_values('IDF', ascending=False)

print("IDF scores (higher = rarer term):")
idf_scores

In [ ]:
# For each document, list the top 5 most important terms by TF-IDF score.
# These are the words that best characterize each document.
print("Top terms per document by TF-IDF score:\n")
feature_names = tfidf_vectorizer.get_feature_names_out()
for i in range(len(documents)):
    row = tfidf_matrix[i].toarray().flatten()
    # Sort descending and take top 5
    top_indices = row.argsort()[::-1][:5]
    top_terms = [(feature_names[j], round(row[j], 4)) for j in top_indices if row[j] > 0]
    print(f"Document {i+1}: {top_terms}")

## Conclusion

We performed the complete text-analytics preprocessing pipeline:

| Step | Purpose |
|------|---------|
| **Tokenization** | Break text into sentences and words |
| **POS Tagging** | Label each word with its grammatical role |
| **Stop Words Removal** | Drop common low-information words |
| **Stemming** | Reduce words to crude root form (Porter Stemmer) |
| **Lemmatization** | Reduce words to dictionary base form (WordNet) |
| **TF-IDF** | Convert text into numeric vectors that highlight distinctive terms |

**Stemming vs Lemmatization** — Stemming is faster but produces non-words; lemmatization is slower but produces real words. Use lemmatization when accuracy matters; stemming for high-throughput tasks.

**TF-IDF intuition** — Common words score low (frequent everywhere), rare words score high (appear in few documents). The result is a numeric vector per document that captures *what makes this document distinctive*. These vectors feed directly into classification, similarity search, and clustering algorithms.

Sure — here's the full conceptual breakdown plus a comprehensive viva question bank for the Text Analytics practical.

---

# Text Analytics — What We Actually Did

**Big picture:** Up until now, all six previous notebooks worked with **structured data** — tables of numbers and categories that fit neatly into rows and columns. Spreadsheets, basically. This notebook is fundamentally different: we're working with **unstructured data** — raw human-written text. There's no "column" in a paragraph. The computer sees `"Text analytics is the process of deriving meaningful information"` as just a long string of characters; it has no idea where words begin and end, what's a noun, what's important, or what makes one sentence different from another.

So the entire job of this notebook is to take **shapeless, messy text** and convert it into **clean numerical vectors** that machine learning algorithms can actually consume. Once we have those vectors, we can do classification (is this email spam?), similarity search (find documents like this one), clustering (group similar articles), sentiment analysis, recommendation systems — basically every text-related ML task you've ever heard of.

The notebook splits into two halves:

- **Part 1 — Preprocessing**: clean and normalize the raw text (5 sub-steps: tokenization → POS tagging → stop word removal → stemming → lemmatization)
- **Part 2 — Vectorization**: turn the cleaned text into numbers using **TF-IDF**

Together, these two halves are called the **NLP pipeline**, and almost every text-processing system on Earth starts with some version of these steps.

---

## Why preprocess text at all?

Raw text is noisy in ways that confuse algorithms:

1. **Punctuation isn't meaningful** — "running," and "running" are the same word but look different to a computer.
2. **Capitalization isn't meaningful** — "Text" and "text" should be treated as one word.
3. **Most words are junk** — words like "the", "is", "of", "and" appear in literally every English sentence. They tell you nothing about what a document is *about*.
4. **Same word, many forms** — "run", "running", "ran", "runs" all refer to the same concept, but a naïve algorithm sees four different tokens.
5. **Computers can't compare strings meaningfully** — there's no math you can do on the word "analytics". You need numbers.

Preprocessing fixes points 1–4. Vectorization (TF-IDF) fixes point 5.

---

## What we did, step by step

### Step 1 — Brought in our toolbox

Loaded **NLTK** (Natural Language Toolkit), the most popular Python library for classical NLP. Downloaded NLTK's language-data files (tokenizer models, stop word lists, POS tagger, WordNet dictionary). Also pulled in **scikit-learn** for the TF-IDF vectorizer and **pandas** for displaying results in tidy tables.

### Step 2 — Defined a sample document

A four-sentence paragraph about text analytics itself. Small enough to inspect manually, big enough to demonstrate every concept.

### Step 3 — Tokenization

**Tokenization = breaking text into smaller units called tokens.** This is the very first thing any NLP pipeline does, because without it the computer is just staring at one giant string.

We did it at **two levels**:

- **Sentence tokenization** — breaks the paragraph into individual sentences. NLTK's `sent_tokenize()` is smarter than just splitting on periods — it knows "Mr." or "U.S.A." aren't sentence endings.
- **Word tokenization** — breaks each sentence into individual words and punctuation marks. NLTK's `word_tokenize()` correctly handles contractions ("don't" → ["do", "n't"]) and treats punctuation as separate tokens.

After this step, a sentence like "Text analytics is widely used." becomes the list `['Text', 'analytics', 'is', 'widely', 'used', '.']`.

### Step 4 — POS Tagging (Part-of-Speech Tagging)

**POS tagging = labeling each word with its grammatical role.** Is this word a noun, a verb, an adjective, a preposition?

NLTK uses the **Penn Treebank tag set** — a standard list of about 36 tags. The most important ones to know:
- **NN** = noun ("text", "process")
- **VB / VBZ / VBG** = verb / 3rd-person verb / gerund ("involve" / "involves" / "involving")
- **JJ** = adjective ("meaningful", "natural")
- **RB** = adverb ("widely", "efficiently")
- **DT** = determiner ("the", "a", "this")
- **IN** = preposition or conjunction ("of", "in", "from")

**Why do we care?** Two reasons:
1. **POS tags help with lemmatization** — knowing "running" is a verb tells the lemmatizer to return "run", not "running" (which it would assume is a noun by default).
2. **POS tags are themselves useful features** for tasks like named entity recognition, syntactic parsing, and grammar checking.

### Step 5 — Stop Words Removal

**Stop words = ultra-common words that carry no real meaning.** "The", "is", "a", "and", "of", "to", "in" — these appear in every English document, so they don't help distinguish documents from each other.

NLTK ships with a built-in list of about 180 English stop words. We:
1. Lowercased every word (so "The" matches "the" in the stop word list)
2. Filtered out any word in the stop word list
3. Also dropped pure punctuation tokens by keeping only words where `isalpha()` is True

This typically cuts the token count by 40–60%. From our paragraph, we went from 50+ raw tokens to about 25 meaningful ones.

**One nuance worth knowing:** stop word removal isn't always the right move. For sentiment analysis, the word "not" matters enormously — "not good" is the opposite of "good", but if you remove "not" as a stop word, both phrases look identical. So always think about whether the words you're removing actually matter for *your* task.

### Step 6 — Stemming

**Stemming = chopping off word endings to reduce a word to its stem.** A crude but fast operation.

- "running" → "run"
- "studies" → "studi"
- "happily" → "happili"
- "analyzing" → "analyz"

Notice that some stems aren't real words ("studi", "analyz"). That's the price of speed. The stemmer doesn't know any English; it just applies pattern-based suffix-stripping rules.

We used the **Porter Stemmer**, the most popular English stemming algorithm (originally published in 1980 and still widely used). Other options include the **Snowball Stemmer** (slightly improved Porter) and the **Lancaster Stemmer** (more aggressive).

**Why stem at all?** It collapses word variants into a single token. So "run", "running", "runs", "ran" all become "run", and the document now has one feature instead of four.

### Step 7 — Lemmatization

**Lemmatization = reducing a word to its base dictionary form (lemma) using actual linguistic knowledge.** Slower but more accurate than stemming.

- "running" → "run" (real word)
- "studies" → "study" (real word, not "studi")
- "better" → "good" (recognizes irregular forms)
- "mice" → "mouse"

The lemmatizer uses **WordNet**, a large lexical database that knows English vocabulary, parts of speech, and morphology. It can only return real dictionary words — never gibberish.

**Critical detail:** the lemmatizer needs to know the part of speech to do its job correctly.
- `lemmatize("running")` (default = noun) → "running" (no change, treats it as a noun)
- `lemmatize("running", pos='v')` (verb) → "run" ✓

This is why POS tagging often happens *before* lemmatization in a real pipeline.

### Stemming vs Lemmatization — when to use each

| | Stemming | Lemmatization |
|---|---|---|
| Speed | Fast | Slower |
| Output | May not be a real word | Always a real word |
| Accuracy | Lower | Higher |
| Needs vocabulary? | No | Yes (WordNet) |
| Use case | Search engines, large-scale text processing | Anything where accuracy matters: chatbots, document analysis |

Rule of thumb: if you're processing millions of documents and just need rough matching, stem. If you're doing anything where word identity matters, lemmatize.

### Steps 8 — Term Frequency and Inverse Document Frequency (TF-IDF)

This is the core of Part 2 — turning words into numbers.

**The problem we're solving:** how do we tell which words *matter* in a document? If we just count word occurrences, the most common word will be "the" — and "the" tells us nothing about the topic. We need a way to weight words by their **distinctiveness**, not just their frequency.

That's exactly what TF-IDF does. It has two parts.

#### Term Frequency (TF) — "how often does this word appear here?"

$$TF(t, d) = \frac{\text{count of term } t \text{ in document } d}{\text{total terms in document } d}$$

If "analytics" appears 3 times in a 30-word document, its TF is 3/30 = 0.1.

But TF alone has a problem: "the" might have a higher TF than "analytics" simply because "the" appears more often. So we need a way to *discount* common words.

#### Inverse Document Frequency (IDF) — "how rare is this word across all documents?"

$$IDF(t) = \log\left(\frac{N}{df(t)}\right)$$

where:
- **N** = total number of documents
- **df(t)** = how many documents contain the term t

If a term appears in **every** document (like "the"), IDF = log(N/N) = log(1) = 0 — its IDF score is zero.
If a term appears in only **one** document out of 1000, IDF = log(1000/1) = high score.

So IDF rewards rare, distinctive terms and punishes common, vague terms.

#### TF-IDF — combining the two

$$TF\text{-}IDF(t, d) = TF(t, d) \times IDF(t)$$

A high TF-IDF score means the word is **frequent in this document** AND **rare across the corpus**. That's the textbook definition of a "distinctive" word — exactly what we want for finding the topic of a document.

#### What we did with TF-IDF in the notebook

1. **Built a corpus** of 4 documents (the 4 sentences from our paragraph).
2. **Computed raw counts** with `CountVectorizer` — a simple matrix where each cell shows how many times a term appears in a document.
3. **Computed TF-IDF** with `TfidfVectorizer` — the same matrix but with each cell weighted by the IDF formula.
4. **Inspected the IDF scores** — sorted from highest (rarest, most distinctive) to lowest (most common).
5. **Found the top 5 terms per document** by TF-IDF score — these are the words that best summarize each sentence.

The end result: each document is now a **vector of numbers**. Document 1 might look like `[0, 0.34, 0.51, 0.0, 0.27, 0.0, ...]` — one number per term in the vocabulary. These vectors are what feed into classifiers, similarity calculators, and clustering algorithms.

---

## How this notebook fits into the bigger picture

We've now covered seven notebooks total:

1. **Data Wrangling I** — clean structurally messy data (missing values, types, encoding)
2. **Data Wrangling II** — clean statistically messy data (outliers, skewness)
3. **Descriptive Statistics** — summarize data
4. **Linear Regression** — predict a continuous number
5. **Logistic Regression** — predict a binary class
6. **Naïve Bayes** — predict a multi-class label
7. **Text Analytics (this notebook)** — preprocess unstructured text and convert it to numeric vectors

Notebook 7 is special because it deals with a fundamentally different kind of input. But the output (numeric vectors) feeds straight back into the techniques from notebooks 4–6. The classic example: take a corpus of product reviews → preprocess and TF-IDF them → run logistic regression on the vectors with target = "positive review" or "negative review" → you've built a sentiment classifier. That's literally how spam filters, sentiment analyzers, and search engines work under the hood.

---

# Possible Viva Questions (with concise answers)

### Conceptual — Text Analytics in General

**Q1. What is text analytics?**
Text analytics is the process of deriving meaningful information from natural language text using techniques like tokenization, POS tagging, stemming, lemmatization, and vectorization (TF-IDF, word embeddings, etc.).

**Q2. What's the difference between structured and unstructured data?**
Structured data has a fixed schema — rows and columns of numbers/categories (CSV, SQL tables). Unstructured data has no schema — raw text, images, audio. Most real-world data (~80%) is unstructured.

**Q3. What is NLP?**
Natural Language Processing — a subfield of AI focused on enabling computers to understand, interpret, and generate human language.

**Q4. What is the typical NLP pipeline?**
Text input → Tokenization → Stop words removal → Stemming/Lemmatization → POS tagging → Vectorization (TF-IDF or embeddings) → Modeling (classification, clustering, etc.).

**Q5. What is a corpus?**
A collection of documents used for analysis. In this notebook our corpus had 4 documents (the 4 sentences).

**Q6. What is a token? What is a document?**
A **token** is a single unit of text — usually a word or punctuation mark. A **document** is a unit of text being analyzed (could be a sentence, paragraph, article, or book — depends on the application).

### Tokenization

**Q7. What is tokenization?**
The process of breaking text into smaller units called tokens. In NLTK, `word_tokenize()` and `sent_tokenize()` perform word-level and sentence-level tokenization respectively.

**Q8. What's the difference between sentence and word tokenization?**
Sentence tokenization splits a document into sentences; word tokenization splits a sentence (or document) into individual words and punctuation.

**Q9. Why is splitting on whitespace not enough for tokenization?**
Because real text has contractions ("don't"), punctuation attached to words ("hello,"), abbreviations ("U.S.A."), and special cases that need linguistic rules to handle correctly.

### POS Tagging

**Q10. What is POS tagging?**
Part-of-Speech tagging — labeling each word with its grammatical category (noun, verb, adjective, etc.).

**Q11. What tag set does NLTK use?**
Penn Treebank — the standard tag set with about 36 tags. Common ones: NN (noun), VB (verb), JJ (adjective), RB (adverb), DT (determiner), IN (preposition).

**Q12. Why is POS tagging useful?**
Two main reasons: (1) it helps lemmatization work correctly (knowing "running" is a verb returns "run"), and (2) POS tags themselves are useful features for tasks like named entity recognition and syntactic parsing.

**Q13. What does "VBG" mean in Penn Treebank?**
Verb in gerund/present-participle form, e.g., "running", "analyzing", "studying".

### Stop Words

**Q14. What are stop words?**
Extremely common words ("the", "is", "and", "of", "in") that carry little distinguishing information and are usually removed before analysis.

**Q15. Why do we remove stop words?**
Because they appear in nearly every document, they don't help distinguish documents from each other, and they bloat the feature space without adding signal.

**Q16. When should we NOT remove stop words?**
When the words actually matter for the task. Sentiment analysis is the classic example — "not good" and "good" mean opposite things, but if you remove "not" as a stop word, they look identical.

**Q17. How many English stop words are in NLTK?**
About 180 (the exact number can vary slightly by NLTK version).

### Stemming

**Q18. What is stemming?**
Reducing a word to its stem (root form) by chopping off suffixes. The result may not be a real dictionary word.

**Q19. What is the Porter Stemmer?**
The most popular English stemming algorithm (Martin Porter, 1980). It applies a series of suffix-stripping rules in stages.

**Q20. Give an example where stemming produces a non-word.**
"studies" → "studi", "analyzing" → "analyz", "happily" → "happili" — none of these are real English words.

**Q21. Other types of stemmers?**
Snowball (improved Porter, supports multiple languages), Lancaster (more aggressive), Regex-based stemmers.

### Lemmatization

**Q22. What is lemmatization?**
Reducing a word to its base dictionary form (lemma) using vocabulary and morphological analysis. The result is always a real word.

**Q23. What's the difference between stemming and lemmatization?**
Stemming uses crude suffix-stripping rules — fast but may produce non-words. Lemmatization uses a vocabulary (like WordNet) and morphological rules — slower but always returns a valid word.

**Q24. What is WordNet?**
A large English lexical database that groups words into synonym sets and tracks their definitions, parts of speech, and relationships. NLTK's lemmatizer uses WordNet.

**Q25. Why does lemmatization need POS information?**
Because the same word can lemmatize differently depending on its role. "running" as a verb lemmatizes to "run"; "running" as a noun (as in "a daily running") stays "running". Without POS, the lemmatizer defaults to noun and may miss the right answer.

**Q26. Give a case where stemming and lemmatization differ.**
- "studies" → stem: "studi", lemma: "study"
- "better" → stem: "better", lemma: "good"
- "was" → stem: "wa", lemma: "be"

**Q27. Which is faster — stemming or lemmatization? Why?**
Stemming is faster because it just applies pattern-based rules. Lemmatization is slower because it has to look words up in a dictionary and (ideally) consider POS tags.

### TF-IDF

**Q28. What does TF-IDF stand for?**
Term Frequency — Inverse Document Frequency.

**Q29. Define Term Frequency.**
TF(t, d) = (count of term t in document d) / (total terms in document d). It measures how often a term appears in a document, normalized by document length.

**Q30. Define Inverse Document Frequency.**
IDF(t) = log(N / df(t)), where N = total number of documents and df(t) = number of documents containing term t. It measures how rare a term is across the corpus.

**Q31. Define TF-IDF.**
TF-IDF(t, d) = TF(t, d) × IDF(t). High score = term is frequent in this document AND rare across the corpus = distinctive of this document.

**Q32. Why is the logarithm used in IDF?**
To dampen the effect of extremely rare terms. Without log, a term appearing in 1 out of 10,000 documents would have an IDF of 10,000 — wildly disproportionate. The log compresses this to a more reasonable scale.

**Q33. What is the IDF of a word that appears in every document?**
log(N/N) = log(1) = 0. So its TF-IDF will also be zero, regardless of how often it appears — exactly the right behavior for stop-word-like terms.

**Q34. What is the IDF of a word that appears in only one document?**
log(N/1) = log(N) — the maximum possible IDF for that corpus. Such a term is highly distinctive.

**Q35. Why is TF-IDF better than just raw counts?**
Raw counts give common words like "the" the highest scores, which is uninformative. TF-IDF down-weights common terms and highlights distinctive ones, producing vectors that actually reflect what each document is about.

**Q36. What is the difference between CountVectorizer and TfidfVectorizer?**
CountVectorizer produces a matrix of raw counts (Term Frequency without normalization). TfidfVectorizer produces a matrix of TF-IDF scores (counts weighted by IDF, with normalization).

**Q37. What is a document-term matrix?**
A 2D matrix where rows = documents, columns = terms (vocabulary), and each cell = count or TF-IDF score of that term in that document.

**Q38. What does fit_transform() do for a vectorizer?**
`fit()` learns the vocabulary from the corpus; `transform()` converts text into the numeric matrix using that vocabulary. `fit_transform()` does both in one step.

**Q39. What's the limitation of TF-IDF?**
It's a "bag of words" representation — it ignores word order, context, and semantics. "Dog bites man" and "Man bites dog" produce identical TF-IDF vectors, even though they mean opposite things. Modern approaches (word embeddings, BERT, transformers) address this.

**Q40. What comes after TF-IDF in a typical NLP project?**
Feed the TF-IDF vectors into an ML model: classification (Naïve Bayes, logistic regression, SVM), clustering (K-means), similarity search (cosine similarity), or topic modeling (LDA).

### Practical / Code

**Q41. What does `nltk.download()` do?**
Downloads NLTK's data files (tokenizer models, corpora, stop word lists) — required because NLTK doesn't bundle them with the library install.

**Q42. What does `isalpha()` do in our stop-word filter?**
Returns True only if the token is purely alphabetic. We use it to drop punctuation tokens (".", ",", "!") that survive tokenization.

**Q43. Why did we lowercase before checking against stop words?**
Because NLTK's stop word list is in lowercase. Without lowercasing, "The" wouldn't match the stop word "the" and would slip through.

**Q44. Why split the paragraph into multiple "documents" for TF-IDF?**
TF-IDF needs more than one document to compute IDF. With a single document, every term's IDF would be log(1/1) = 0 and TF-IDF would collapse. So we treated each sentence as a separate document.

**Q45. What does `get_feature_names_out()` return?**
The list of vocabulary terms (column labels) corresponding to the TF-IDF matrix. Each column index maps to one term.

### Higher-Level / Application

**Q46. Where is TF-IDF used in industry?**
Search engines (ranking documents by relevance to a query), spam filters, document classification, recommendation systems, plagiarism detection, automatic keyword extraction.

**Q47. What's the "bag of words" model?**
A representation that treats a document as an unordered multiset of words, ignoring grammar and word order. TF-IDF is a bag-of-words technique.

**Q48. What are the alternatives to TF-IDF?**
Word embeddings (Word2Vec, GloVe, FastText), contextual embeddings (BERT, GPT), and document embeddings (Doc2Vec). These capture semantic meaning that TF-IDF misses.

**Q49. What is sentiment analysis?**
Determining the emotional tone of text (positive/negative/neutral). A common application: classify product reviews or social media posts.

**Q50. How would you build a spam filter using what we did today?**
1. Collect labeled emails (spam vs not-spam). 2. Preprocess each email (tokenize, remove stop words, lemmatize). 3. Convert to TF-IDF vectors. 4. Train a classifier (Naïve Bayes works famously well for this). 5. For new emails: preprocess → vectorize with the same fitted vectorizer → predict.

**Q51. What is named entity recognition (NER)?**
Identifying and classifying named entities in text — people, organizations, locations, dates, etc. It's a common downstream NLP task; uses POS tags and other features.

**Q52. What is the difference between bag-of-words and n-grams?**
Bag-of-words treats each word as a separate feature, ignoring order. N-grams treat sequences of N consecutive words as features. "New York" as a 2-gram (bigram) is different from "New" + "York" as separate tokens — useful when word combinations matter.

---

That's the full breakdown — concepts, steps, and 52 viva questions covering every angle of the practical. If your examiner asks something not on this list, the universal fallback answer is the pipeline itself: **tokenize → tag → remove stop words → stem/lemmatize → vectorize with TF-IDF → ready for ML**. That's the spine of every NLP project, and knowing it cold means you can derive most other answers from there.